In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_interaction_qc(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "InteractionId	string,"
        #     "SourceSystem	string,"
        #     "PortfolioKey	string,"
        #     "PortfolioName	string,"
        #     "IngestionTimestamp	timestamp,"
        #     "BatchId	string,"
        #     "InteractionKey	string,"
        #     "InteractionCode	int,"
        #     "ExposureKey	string,"
        #     "PartyKey	int,"
        #     "CDI	string,"
        #     "InteractionTypeCode	string,"
        #     "InteractionTypeKey	string,"
        #     "InteractionType	string,"
        #     "InteractionGroupTypeCode	int,"
        #     "InteractionGroupType	string,"
        #     "InteractionCategoryCode	int,"
        #     "InteractionCategory	string,"
        #     "DialerTypeCode	string,"
        #     "DialerType	string,"
        #     "ContactTypeCode	string,"
        #     "ContactType	string,"
        #     "AttemptTypeCode	string,"
        #     "AttemptType	string,"
        #     "AttemptOutcomeCode	string,"
        #     "AttemptOutcome	string,"
        #     "AttemptSpecialMetricCode	string,"
        #     "AttemptSpecialMetric	string,"
        #     "IsPositiveFlag	boolean,"
        #     "IsPromiseFlag	boolean,"
        #     "IsActiveFlag	boolean,"
        #     "IsLegalFlag	boolean,"
        #     "IsCallFlag	boolean,"
        #     "IsFtFAppointmentFlag	boolean,"
        #     "IsRPCFlag	boolean,"
        #     "IsRPCFlagCalc	boolean,"
        #     "LegalProcessStatusCode	int,"
        #     "LegalProcessStatus	string,"
        #     "LegalProcessStatusPriority	string,"
        #     "BaseCategory	string,"
        #     "FirstLevelCategory	string,"
        #     "SecondLevelCategory	string,"
        #     "ThirdLevelCategory	string,"
        #     "NotificationType	string,"
        #     "Hierarchy	int,"
        #     "InteractionDate	timestamp,"
        #     "InteractionComments	string,"
        #     "ResultComments	string,"
        #     "DebtorStatus	string,"
        #     "DebtorStatusComments	string,"
        #     "CurrentStrategy	string,"
        #     "CategoryOfAction	string,"
        #     "SubCategoryOfAction	string,"
        #     "Comments	string,"        
        #     "CaseCode	int,"
        #     "CaseCodePrimary	string,"
        #     "ContactPerson	string,"
        #     "ContactPersonCategoryCode	int,"
        #     "ContactPersonCategory	int,"
        #     "InteractionPhone	string,"
        #     "ExternalCompanyKey	int,"
        #     "CreatedDate	timestamp,"
        #     "CreatedBy	string,"
        #     "LastUpdatedDate	date,"
        #     "LastUpdatedBy	string,"
        #     "PromiseKey	int,"
        #     "PromiseAmount	decimal(19,4),"
        #     "PromiseTypeCode	int,"
        #     "PromiseType	int,"
        #     "PromiseToPayTypeCode	int,"
        #     "PromiseToPayType	string,"
        #     "PromiseDate	timestamp,"
        #     "PromiseDueDate	string,"
        #     "PromiseStatusCode	int,"
        #     "PromiseStatus	string,"
        #     "PromiseToPayAmount	decimal(19,4),"
        #     "PromiseComments	string,"
        #     "PaymentKey	int,"
        #     "PromiseLastUpdatedDate	timestamp,"
        #     "PromiseLastUpdatedBy	string,"
        #     "CoverageOfPaymentVSPromiseAmount	decimal(19,4),"
        #     "CoverageOfPromiseVSPaymentAmount	int,"
        #     "DaysBetweenPromiseAndDueDate	int,"
        #     "ScheduleDate date,"
        #     "BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Interaction"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qc",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")
        # ================== activity ==================

        activity_extra_filters = None

        if activity_extra_filters is not None:
            activity_extra_filters.append(date_filter)
            activity_final_filters = " AND ".join(activity_extra_filters)
        else:
            activity_final_filters = date_filter

        activity_selected_fields = [
            "cac_code",
            "contact_type",
            "vend_code",
            "cac_date",
            "comments",
            "case_code",
            "act_phone",
            "extc_code",
            "cac_created",
            "cac_created_by",
            "cac_last_upd",
            "sort_code",
            "ACTV_CODE",
            "businessdate",
        ]

        activity = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_activity",
            activity_final_filters,
            activity_selected_fields,
        )
        # ================== case_master ==================

        case_master_extra_filters = None

        if case_master_extra_filters is not None:
            case_master_extra_filters.append(date_filter)
            case_master_final_filters = " AND ".join(case_master_extra_filters)
        else:
            case_master_final_filters = date_filter

        case_master_selected_fields = [
            "account_code",
            "cust_code",
            "old_case_code",
            "qc_portfolio",
            "case_code",
            "case_code_primary",
        ]

        case_master = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_case_master",
            case_master_final_filters,
            case_master_selected_fields,
        )
        # ================== activities_vw ==================

        activities_vw_extra_filters = None

        if activities_vw_extra_filters is not None:
            activities_vw_extra_filters.append(date_filter)
            activities_vw_final_filters = " AND ".join(activities_vw_extra_filters)
        else:
            activities_vw_final_filters = date_filter

        activities_vw_selected_fields = [
            "ACTV_TYPE_CODE",
            "ACTV_DAT",
            "ACTV_NAME",
            "attempt_type",
            "pos_neg_oth",
            "special_metric",
            "ACTV_ISPROMISE",
            "ACTV_ACTIVE",
            "ACTV_IS_LEGAL",
            "ACTV_INTCODE",
        ]

        activities_vw = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_activities_vw",
            activities_vw_final_filters,
            activities_vw_selected_fields,
        )
        # ================== customer ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "cdi",
            "cust_code",
        ]

        customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_customer",  # check name
            customer_final_filters,
            customer_selected_fields,
        )
        # ================== lps_priority ==================

        lps_priority_extra_filters = None

        if lps_priority_extra_filters is not None:
            lps_priority_extra_filters.append(date_filter)
            lps_priority_final_filters = " AND ".join(lps_priority_extra_filters)
        else:
            lps_priority_final_filters = date_filter

        lps_priority_selected_fields = [
            "LPS_code",
            "short_code",
            "LPS",
            "priority",
        ]

        lps_priority = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_lps_priority",
            lps_priority_final_filters,
            lps_priority_selected_fields,
        )
        # ================== promise ==================

        promise_extra_filters = None

        if promise_extra_filters is not None:
            promise_extra_filters.append(date_filter)
            promise_final_filters = " AND ".join(promise_extra_filters)
        else:
            promise_final_filters = date_filter

        promise_selected_fields = [
            "cupr_pay_amount",
            "comments",
            "cupr_date",
            "cupr_due_date",
            "cupr_status_code",
            "cupr_code",
            "cupr_amount",
            "cupr_pay_type",
            "cac_code",
        ]

        promise = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_promise",
            promise_final_filters,
            promise_selected_fields,
        )
        # ================== lst_of_val ==================

        lst_of_val_extra_filters = None

        if lst_of_val_extra_filters is not None:
            lst_of_val_extra_filters.append(date_filter)
            lst_of_val_final_filters = " AND ".join(lst_of_val_extra_filters)
        else:
            lst_of_val_final_filters = date_filter

        lst_of_val_selected_fields = [
            "lov_desc",
            "lov_code",
            "lovt_desc",
        ]

        lst_of_val = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_lst_of_val",
            lst_of_val_final_filters,
            lst_of_val_selected_fields,
        )
        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "vendor_code_qualco",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== Joins ==================

        LstOfVal_ContactType = lst_of_val.alias("LstOfVal_ContactType")
        LstOfVal_ActivityType = lst_of_val.alias("LstOfVal_ActivityType")
        LstOfVal_DialerActType = lst_of_val.alias("LstOfVal_DialerActType")
        LstOfVal_PromState = lst_of_val.alias("LstOfVal_PromState")
        LstOfVal_PayType = lst_of_val.alias("LstOfVal_PayType")

        LstOfVal_ContactType = (
            LstOfVal_ContactType.withColumnRenamed("lovt_desc", "lovt_desc_ContactType")
            .withColumnRenamed("lov_code", "lov_code_ContactType")
            .withColumnRenamed("lov_desc", "lov_desc_ContactType")
        )
        LstOfVal_ActivityType = (
            LstOfVal_ActivityType.withColumnRenamed(
                "lovt_desc", "lovt_desc_ActivityType"
            )
            .withColumnRenamed("lov_code", "lov_code_ActivityType")
            .withColumnRenamed("lov_desc", "lov_desc_ActivityType")
        )
        LstOfVal_DialerActType = (
            LstOfVal_DialerActType.withColumnRenamed(
                "lovt_desc", "lovt_desc_DialerActType"
            )
            .withColumnRenamed("lov_code", "lov_code_DialerActType")
            .withColumnRenamed("lov_desc", "lov_desc_DialerActType")
        )
        LstOfVal_PromState = (
            LstOfVal_PromState.withColumnRenamed("lovt_desc", "lovt_desc_PromState")
            .withColumnRenamed("lov_code", "lov_code_PromState")
            .withColumnRenamed("lov_desc", "lov_desc_PromState")
        )
        LstOfVal_PayType = (
            LstOfVal_PayType.withColumnRenamed("lovt_desc", "lovt_desc_PayType")
            .withColumnRenamed("lov_code", "lov_code_PayType")
            .withColumnRenamed("lov_desc", "lov_desc_PayType")
        )

        target_table_qc = (
            activity.join(
                activities_vw,
                (activity["sort_code"] == activities_vw["ACTV_INTCODE"]),
                "inner",
            )
            .join(
                case_master,
                (activity["case_code"] == case_master["case_code"]),
                "inner",
            )
            .join(
                customer,
                (case_master["cust_code"] == customer["cust_code"]),
                "inner",
            )
            .join(
                promise,
                (activity["cac_code"] == promise["cac_code"]),
                "left",
            )
            .join(
                lps_priority,
                (activities_vw["ACTV_INTCODE"] == lps_priority["short_code"]),
                "left",
            )
            .join(
                LstOfVal_ContactType,
                (
                    activity["contact_type"]
                    == LstOfVal_ContactType["lov_code_ContactType"]
                )
                & (
                    LstOfVal_ContactType["lovt_desc_ContactType"]
                    == "ACTIVITY_CONTACT_TYPE"
                ),
                "left",
            )
            .join(
                LstOfVal_ActivityType,
                (
                    activities_vw["ACTV_TYPE_CODE"]
                    == LstOfVal_ActivityType["lov_code_ActivityType"]
                )
                & (LstOfVal_ActivityType["lovt_desc_ActivityType"] == "ACTIVITY_TYPES"),
                "left",
            )
            .join(
                LstOfVal_DialerActType,
                (
                    activities_vw["actv_dat"]
                    == LstOfVal_DialerActType["lov_code_DialerActType"]
                )
                & (
                    LstOfVal_DialerActType["lovt_desc_DialerActType"]
                    == "DIALER_ACTIVITY_TYPE"
                ),
                "left",
            )
            .join(
                LstOfVal_PromState,
                (
                    promise["cupr_status_code"]
                    == LstOfVal_PromState["lov_code_PromState"]
                )
                & (LstOfVal_PromState["lovt_desc_PromState"] == "PROM_STATE"),
                "left",
            )
            .join(
                LstOfVal_PayType,
                (promise["cupr_pay_type"] == LstOfVal_PayType["lov_code_PayType"])
                & (LstOfVal_PayType["lovt_desc_PayType"] == "PAY_TYPE"),
                "left",
            )
            .join(
                portfolio,
                (activity["vend_code"] == portfolio["vendor_code_qualco"]),
                "left",
            )
        )

        # HashKeys
        target_table_qc_InteractionID_primary_key_fields = [
            activity.cac_code,
            promise.cupr_code,
            lit("qc"),
        ]

        InteractionKey_primary_key_fields = [
            activity.cac_code,
            promise.cupr_code,
        ]

        IsRPCFlag_type_condition = when(
            activities_vw["attempt_type"] == "RPC", lit("1")
        ).otherwise(lit("0"))

        target_table_qc = (
            target_table_qc.withColumn(
                "InteractionId",
                sha2(
                    concat_ws(";", *target_table_qc_InteractionID_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "InteractionKey", concat_ws(";", *InteractionKey_primary_key_fields)
            )
            .withColumn("IsRPCFlag", IsRPCFlag_type_condition)
        ).distinct()

        # ================== Selection of Final Columns ==================

        target_table_qc = target_table_qc.select(
            target_table_qc.InteractionId.cast("string"),
            lit("qc").alias("SourceSystem"),
            target_table_qc.vend_code.cast("string").alias("PortfolioKey"),
            target_table_qc.PortfolioName.cast("string"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            target_table_qc.InteractionKey,
            activity.cac_code.alias("InteractionCode"),
            target_table_qc.old_case_code.cast("string").alias("ExposureKey"),
            case_master.cust_code.alias("PartyKey"),
            customer.cdi.cast("string").alias("CDI"),
            target_table_qc.sort_code.cast("string").alias("InteractionTypeCode"),
            target_table_qc.ACTV_INTCODE.cast("string").alias("InteractionTypeKey"),
            target_table_qc.ACTV_NAME.alias("InteractionType"),
            lit(None).alias("InteractionGroupTypeCode"),
            lit(None).alias("InteractionGroupType"),
            target_table_qc.ACTV_TYPE_CODE.alias("InteractionCategoryCode"),
            LstOfVal_ActivityType.lov_desc_ActivityType.alias("InteractionCategory"),
            target_table_qc.ACTV_DAT.cast("string").alias("DialerTypeCode"),
            LstOfVal_DialerActType.lov_desc_DialerActType.alias("DialerType"),
            target_table_qc.contact_type.cast("string").alias("ContactTypeCode"),
            LstOfVal_ContactType.lov_desc_ContactType.alias("ContactType"),
            target_table_qc.attempt_type.alias("AttemptTypeCode"),
            target_table_qc.attempt_type.alias("AttemptType"),
            target_table_qc.pos_neg_oth.alias("AttemptOutcomeCode"),
            target_table_qc.pos_neg_oth.alias("AttemptOutcome"),
            target_table_qc.special_metric.alias("AttemptSpecialMetricCode"),
            target_table_qc.special_metric.alias("AttemptSpecialMetric"),
            lit(None).cast("boolean").alias("IsPositiveFlag"),
            target_table_qc.ACTV_ISPROMISE.cast("boolean").alias("IsPromiseFlag"),
            target_table_qc.ACTV_ACTIVE.cast("boolean").alias("IsActiveFlag"),
            target_table_qc.ACTV_IS_LEGAL.cast("boolean").alias("IsLegalFlag"),
            lit(None).cast("boolean").alias("IsCallFlag"),
            lit(None).cast("boolean").alias("IsFtFAppointmentFlag"),
            target_table_qc.IsRPCFlag.cast("boolean").alias("IsRPCFlag"),
            lit(None).cast("boolean").alias("IsRPCFlagCalc"),
            target_table_qc.LPS_code.alias("LegalProcessStatusCode"),
            target_table_qc.LPS.cast("string").alias("LegalProcessStatus"),
            target_table_qc.priority.cast("string").alias("LegalProcessStatusPriority"),
            lit(None).alias("BaseCategory"),
            lit(None).alias("FirstLevelCategory"),
            lit(None).alias("SecondLevelCategory"),
            lit(None).alias("ThirdLevelCategory"),
            lit(None).alias("NotificationType"),
            lit(None).cast("int").alias("Hierarchy"),
            target_table_qc.cac_date.cast("timestamp").alias("InteractionDate"),
            activity.comments.alias("InteractionComments"),
            lit(None).alias("ResultComments"),
            lit(None).alias("DebtorStatus"),
            lit(None).alias("CurrentStrategy"),
            lit(None).alias("CategoryOfAction"),
            lit(None).alias("DebtorStatusComments"),
            lit(None).alias("SubCategoryOfAction"),
            lit(None).alias("Comments"),
            activity.case_code.alias("CaseCode"),
            target_table_qc.case_code_primary.cast("string").alias("CaseCodePrimary"),
            lit(None).alias("ContactPerson"),
            lit(None).alias("ContactPersonCategoryCode"),
            lit(None).alias("ContactPersonCategory"),
            target_table_qc.act_phone.alias("InteractionPhone"),
            target_table_qc.extc_code.alias("ExternalCompanyKey"),
            target_table_qc.cac_created.alias("CreatedDate"),
            target_table_qc.cac_created_by.alias("CreatedBy"),
            target_table_qc.cac_last_upd.cast("date").alias("LastUpdatedDate"),
            lit(None).alias("LastUpdatedBy"),
            target_table_qc.cupr_code.alias("PromiseKey"),
            target_table_qc.cupr_amount.alias("PromiseAmount"),
            lit(None).alias("PromiseTypeCode"),
            lit(None).alias("PromiseType"),
            target_table_qc.cupr_pay_type.alias("PromiseToPayTypeCode"),
            LstOfVal_PayType.lov_desc_PayType.alias("PromiseToPayType"),
            target_table_qc.cupr_date.cast("timestamp").alias("PromiseDate"),
            target_table_qc.cupr_due_date.cast("string").alias("PromiseDueDate"),
            target_table_qc.cupr_status_code.alias("PromiseStatusCode"),
            LstOfVal_PromState.lov_desc_PromState.alias("PromiseStatus"),
            target_table_qc.cupr_pay_amount.alias("PromiseToPayAmount"),
            promise.comments.alias("PromiseComments"),
            lit(None).alias("PaymentKey"),
            lit(None).alias("PromiseLastUpdatedDate"),
            lit(None).alias("PromiseLastUpdatedBy"),
            lit(None).alias("CoverageOfPaymentVSPromiseAmount"),
            lit(None).alias("CoverageOfPromiseVSPaymentAmount"),
            lit(None).alias("DaysBetweenPromiseAndDueDate"),
            lit(None).alias("ScheduleDate"),
            activity.businessdate.alias("BusinessDate"),
        )
        # ================== Append ==================

        target_table_qc.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_interaction_qc(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)